In [1]:
import json

def load_data(filename: str) -> dict:
    """Load JSON data from file."""
    with open(filename, "r", encoding="utf-8") as f:
        return json.load(f)


def find_people_you_may_know(user_id: int, data: dict):
    """
    Recommend 'people you may know' based on mutual friends.
    """
    # Map each user id to a set of their friends
    user_friends = {
        user["id"]: set(user["friends"])
        for user in data["users"]
    }

    # If user doesn't exist, return empty list
    if user_id not in user_friends:
        return []

    direct_friends = user_friends[user_id]
    suggestions = {}

    # Look at friends-of-friends
    for friend in direct_friends:
        for mutual in user_friends.get(friend, set()):
            # Skip yourself and already-direct friends
            if mutual == user_id or mutual in direct_friends:
                continue
            suggestions[mutual] = suggestions.get(mutual, 0) + 1

    # Sort suggested ids by mutual-friend count (descending)
    sorted_suggestions = sorted(
        suggestions.items(),
        key=lambda x: x[1],
        reverse=True
    )

    return [uid for uid, _ in sorted_suggestions]


# ---------- main usage ----------

data = load_data("massive_data.json")   # filename must match exactly

user_id = 10   # Sneha
recommendations = find_people_you_may_know(user_id, data)
print("People you may know (user IDs):", recommendations)

# Optional: show names instead of only IDs
id_to_name = {u["id"]: u["name"] for u in data["users"]}
named_recs = [id_to_name.get(uid, f"User {uid}") for uid in recommendations]
print("People you may know (names):", named_recs)


People you may know (user IDs): [1, 9, 43, 44, 45, 46]
People you may know (names): ['Amit', 'Ravi', 'User 43', 'User 44', 'User 45', 'User 46']
